# Jaccard Coefficient with Neptune Analytics

This notebook demonstrates using the Jaccard coefficient algorithm via the nx-neptune backend.
The Jaccard coefficient measures the similarity between two nodes by computing the ratio of
shared neighbors to total unique neighbors: |Γ(u) ∩ Γ(v)| / |Γ(u) ∪ Γ(v)|.
It is useful in link prediction — predicting which edges are likely to form in a network.

## Setup and Imports

In [ ]:
# Check the Python version:
from sys import version_info
assert version_info >= (3, 11), "Python 3.11 or higher is required"

import os
import requests
import pandas as pd

import networkx as nx
from nx_neptune import NeptuneGraph
from nx_neptune.clients import Node
from nx_neptune.utils.utils import get_stdout_logger

In [ ]:
logger = get_stdout_logger(__name__,[
                    'nx_neptune.algorithms.link_prediction.jaccard',
                    'nx_neptune.na_graph', 'nx_neptune.utils.decorators',
                    'nx_neptune.instance_management',__name__])

# Ignore cache warnings
nx.config.warnings_to_ignore.add("cache")

## Check for Neptune Analytics Graph ID

In [ ]:
# Read and load graphId from environment variable
graph_id = os.getenv('NETWORKX_GRAPH_ID')

# If not set, you can set it here
if not graph_id:
    # Uncomment and set your Graph ID
    # %env NETWORKX_GRAPH_ID=your-neptune-analytics-graph-id
    # graph_id = os.getenv('NETWORKX_GRAPH_ID')
    print("Warning: Environment Variable NETWORKX_GRAPH_ID is not defined")
    print("You can set it using: %env NETWORKX_GRAPH_ID=your-neptune-analytics-graph-id")
else:
    print(f"Using Neptune Analytics Graph ID: {graph_id}")

## Download and configure Air route dataset

In [ ]:
# Download routes data
routes_url = "https://raw.githubusercontent.com/jpatokal/openflights/master/data/routes.dat"
routes_file = "resources/notebook_test_data_routes.dat"

# Ensure the directory exists
os.makedirs(os.path.dirname(routes_file), exist_ok=True)

# Download only if file doesn't exist
if not os.path.isfile(routes_file):
    with open(routes_file, "wb") as f:
        f.write(requests.get(routes_url).content)

cols = [
    "airline", "airline_id", "source_airport", "source_airport_id",
    "dest_airport", "dest_airport_id", "codeshare", "stops", "equipment",
]
routes = pd.read_csv(routes_file, names=cols)
routes = routes[["source_airport", "dest_airport"]].dropna()

air_route_graph = nx.from_pandas_edgelist(
    routes, source="source_airport", target="dest_airport",
    create_using=nx.Graph()
)
print(f"Graph loaded: {air_route_graph.number_of_nodes()} nodes, {air_route_graph.number_of_edges()} edges")

## Example 1: Basic Jaccard Coefficient

Compute the Jaccard coefficient for a specific pair of airports to determine how similar their route networks are.

In [ ]:
# Compute Jaccard coefficient for specific pairs
pairs = [("JFK", "LAX"), ("SFO", "ORD"), ("ATL", "DFW")]
result = nx.jaccard_coefficient(air_route_graph, ebunch=pairs, backend="neptune")

for u, v, score in result:
    print(f"({u}, {v}) -> {score:.6f}")

## Example 2: Jaccard Coefficient with Edge Label Filtering

Filter the computation to only consider specific edge types when determining neighbor sets.

In [ ]:
# Compute with edge label filtering
pairs = [("JFK", "LAX"), ("SFO", "ORD")]
result = nx.jaccard_coefficient(
    air_route_graph,
    ebunch=pairs,
    backend="neptune",
    edge_labels=["RELATES_TO"]
)

for u, v, score in result:
    print(f"({u}, {v}) -> {score:.6f}")

## Example 3: Jaccard Coefficient with Traversal Direction

Control which direction of edges are considered when computing neighbor sets.

In [ ]:
# Compute with both inbound and outbound edges
pairs = [("JFK", "LAX"), ("ATL", "ORD")]
result = nx.jaccard_coefficient(
    air_route_graph,
    ebunch=pairs,
    backend="neptune",
    traversal_direction="both"
)

for u, v, score in result:
    print(f"({u}, {v}) -> {score:.6f}")